### Analitica de Datos — Universidad de San Andres
### Clase 01: Exploracion, Manipulacion y Visualizacion

**Otoño 2026 · Negocios Digitales**

---

### 

| # | Seccion | Temas |
|---|---|---|
| 1 | Setup | Instalacion e imports |
| 2 | Carga y primer vistazo | `read_csv`, `head`, `tail`, `shape` |
| 3 | Tipos y estadísticas | `info`, `describe`, `value_counts` |
| 4 | Calidad del dato | Nulos, duplicados, limpieza |
| 5 | Manipulacion | Filtros, columnas nuevas, ordenamiento, groupby |
| 6 | Visualizacion | Countplot, histograma, boxplot, scatter, heatmap |

###

---

**Dataset:** Startups de IA con datos reales (Crunchbase / TechCrunch / PitchBook, 2024-2025).
Empresas como OpenAI, Anthropic, xAI, DeepSeek, Mistral, Cursor, Canva, Waymo, Figure AI y mas.

| Columna | Descripcion |
|---|---|
| `startup` | Nombre de la empresa |
| `pais` | Pais de origen |
| `anio_fundacion` | Año de fundacion |
| `sector` | Categoria (LLM, Robotica, Coding AI...) |
| `ultima_ronda` | Etapa de la ultima ronda (Seed, Serie A/B/C...) |
| `monto_ronda_usd_mm` | Monto levantado en la ultima ronda (USD millones) |
| `valuacion_usd_mm` | Valuacion al momento de esa ronda (USD millones) |
| `empleados` | Cantidad de empleados |
| `inversores_clave` | Principales inversores |

###

---

### Setup y Librerias

In [ ]:
!pip install pandas matplotlib seaborn --quiet

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

---
### Carga y Primer Vistazo

In [ ]:
df_startups = pd.read_csv('startups.csv', index_col=0)

In [ ]:
# Primeras 5 filas
df_startups.head()

In [ ]:
# Ultimas 5 filas
df_startups.tail()

In [ ]:
# (filas, columnas)
print(f"El dataset contiene {df_startups.shape[0]} filas y {df_startups.shape[1]} columnas")

In [ ]:
# Nombres de las columnas
df_startups.columns.tolist()

---
### Tipos de dato y estadísticas

In [ ]:
# Tipo de dato de cada columna + cuantos valores no son nulos
df_startups.info()

In [ ]:
# Estadisticas descriptivas de columnas numericas
df_startups.describe().round(2)

`monto_ronda_usd_mm`: media ~USD 740M vs mediana ~USD 231M. Pocas mega-rondas (OpenAI 6.600M, xAI 6.000M, Databricks 10.000M) jalan el promedio hacia arriba. La mediana es el numero honesto.

`valuacion_usd_mm`: media ~USD 14.700M vs mediana ~USD 3.000M — OpenAI (USD 300.000M) es un outlier enorme que distorsiona completamente el promedio.

`empleados`: media 654 vs mediana 250 — mismo patron. Databricks (6.000) y Waymo (4.000) jalan la media.

In [ ]:
# Comparar media vs mediana para ver el sesgo
cols = ['monto_ronda_usd_mm', 'valuacion_usd_mm', 'empleados']

for col in cols:
    media   = df_startups[col].mean()
    mediana = df_startups[col].median()
    ratio   = media / mediana
    print(f"{col}")
    print(f"  Media:   {media:>10,.0f}")
    print(f"  Mediana: {mediana:>10,.0f}")
    print(f"  Ratio:   {ratio:>10.1f}x  <- cuanto mas alto, mas sesgado")
    print()

In [ ]:
# Los 3 mayores de cada columna — los culpables del sesgo
for col in cols:
    print(f"Top 3 por {col}:")
    print(df_startups.nlargest(3, col)[['startup', col]].to_string(index=False))
    print()

In [ ]:
# Cuantas veces aparece cada valor — ordenado de mayor a menor
df_startups['startup'].value_counts()

In [ ]:
# Cuantos valores distintos hay — devuelve un numero
df_startups['startup'].nunique()

In [ ]:
# Cuales son los valores distintos — devuelve la lista
df_startups['startup'].unique()

---
### Calidad del dato 

In [ ]:
# Cuantos nulos hay por columna
df_startups.isnull().sum()

In [ ]:
# Lo mismo pero en porcentaje
(df_startups.isnull().sum() / len(df_startups) * 100).round(1)

In [ ]:
# Tabla de nulos: cantidad y porcentaje
nulos = df_startups.isnull().sum()
porcentaje = (nulos / len(df_startups) * 100).round(1)

tabla_nulos = pd.DataFrame({
    'nulos': nulos,
    '%': porcentaje
})

tabla_nulos[tabla_nulos['nulos'] > 0].sort_values('nulos', ascending=False)


`valuacion_usd_mm` (~27%): DeepSeek es autofinanciada, algunas startups no divulgan valuacion publica.
`monto_ronda_usd_mm` (~10%): Midjourney nunca levanto ronda externa, empresas publicas no reportan monto reciente.

Estos nulos son **datos faltantes reales**, no errores del dataset. Seaborn los ignora automaticamente en los graficos.

In [ ]:
# Ver filas que tienen nulo en valuacion
df_startups[df_startups['valuacion_usd_mm'].isnull()][['startup', 'pais', 'sector', 'ultima_ronda','valuacion_usd_mm']].head()

In [ ]:
# Cuantas filas duplicadas hay
df_startups.duplicated().sum()

In [ ]:
# Ver los duplicados (keep=False muestra TODAS las ocurrencias)
df_startups[df_startups.duplicated(keep=False)].sort_values('startup').head()

In [ ]:
# Eliminar duplicados — conservar la primera ocurrencia
df_clean = df_startups.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Antes : {len(df_startups)} filas")
print(f"Despues: {len(df_clean)} filas")

`keep='first'` conserva la primera ocurrencia de cada fila duplicada y elimina las siguientes.
`reset_index(drop=True)` reinicia el indice desde 0 para que quede limpio.

A partir de aca trabajamos siempre con `df_clean`.

---
### Manipulacion

#### Seleccionar Columnas y Filas

In [ ]:
# Seleccionar Columnas y Filas
df_clean['startup'].head(10)

In [ ]:
# Seleccionar multiples columnas
df_clean[['startup', 'sector', 'valuacion_usd_mm']].head(10)

In [ ]:
# Filtrar filas: solo startups de USA
df_usa = df_clean[df_clean['pais'] == 'USA']
print(f"Startups de USA: {len(df_usa)}")
df_usa[['startup', 'sector', 'valuacion_usd_mm']].head(8)

In [ ]:
# Filtrar con condicion numerica: valuacion mayor a USD 10.000M
df_clean[df_clean['valuacion_usd_mm'] > 10000][['startup', 'pais', 'valuacion_usd_mm']].sort_values('valuacion_usd_mm', ascending=False)

In [ ]:
# Condicion multiple: USA + valuacion conocida
df_clean[(df_clean['pais'] == 'USA') & (df_clean['valuacion_usd_mm'].notna())][['startup', 'sector', 'valuacion_usd_mm']].head(8)

| Método | Devuelve True cuando... | Uso típico |
|---|---|---|
| `.isnull()` | el valor **es** nulo | encontrar filas con datos faltantes |
| `.notna()` | el valor **no es** nulo | filtrar filas con datos completos |

`&` = AND, `|` = OR. Cada condicion debe ir entre parentesis cuando se combinan.
`.notna()` = lo opuesto a `.isnull()` — selecciona filas donde el valor existe.

| Operador | Nombre | Devuelve True cuando... | Ejemplo |
|---|---|---|---|
| `&` | AND | las DOS condiciones son True | `(df['pais'] == 'USA') & (df['valuacion'] > 1000)` |
| `\|` | OR | AL MENOS UNA condición es True | `(df['pais'] == 'USA') \| (df['pais'] == 'UK')` |

In [ ]:
# Filtrar con lista de valores — isin()
paises_top = ['USA', 'UK', 'Israel', 'China', 'Canada']
df_clean[df_clean['pais'].isin(paises_top)][['startup', 'pais', 'sector']]

#### Ordenar

In [ ]:
# Top 10 startups por valuacion
df_clean.sort_values('valuacion_usd_mm', ascending=False).head(10)[['startup', 'pais', 'sector', 'valuacion_usd_mm']]

In [ ]:
# Top 10 por empleados
df_clean.sort_values('empleados', ascending=False).head(10)[['startup', 'sector', 'empleados', 'valuacion_usd_mm']]

#### Crear Columnas Nuevas

In [ ]:
# Edad de la startup al dia de hoy
df_clean['edad_anios'] = 2025 - df_clean['anio_fundacion']
df_clean[['startup', 'anio_fundacion', 'edad_anios']].head(8)

In [ ]:
# Una startup es "unicornio" si su valuacion supera USD 1.000 millones
df_clean['es_unicornio'] = df_clean['valuacion_usd_mm'] > 1000

df_clean[['startup', 'valuacion_usd_mm', 'es_unicornio']].head(10)

In [ ]:
# Cuantos unicornios hay en el dataset?
df_clean['es_unicornio'].value_counts()

#### Grouby - resumir por grupos

In [ ]:
# Promedio de valuacion por sector
df_clean.groupby('sector')['valuacion_usd_mm'].mean().round(1).sort_values(ascending=False).reset_index(drop=False).rename(columns={'valuacion_usd_mm': 'valuacion_promedio_usd_mm'})

In [ ]:
# Resumen completo por sector
resumen_sector = df_clean.groupby('sector').agg(
    cantidad     = ('startup',         'count'),
    valuacion_md = ('valuacion_usd_mm', 'median'),
    monto_md     = ('monto_ronda_usd_mm','median'),
    empleados_md = ('empleados',        'median'),
).round(1).sort_values('cantidad', ascending=False)

resumen_sector

In [ ]:
# Cuantas startups hay por pais
tabla = (df_clean.groupby('pais')['startup']
         .count()
         .sort_values(ascending=False)
         .reset_index()
         .rename(columns={'startup': 'cantidad'}))

tabla

In [ ]:
# Resumen por etapa de ronda
df_clean.groupby('ultima_ronda').agg(
    cantidad     = ('startup',          'count'),
    valuacion_md = ('valuacion_usd_mm',  'median'),
    monto_md     = ('monto_ronda_usd_mm','median'),
).round(1)

`groupby` + `agg` es una de las combinaciones mas poderosas de pandas.
`count` = cuantos hay · `mean` = promedio · `median` = mediana · `max` = maximo · `min` = minimo

La mediana es mas robusta que la media cuando hay outliers — que es exactamente el caso de valuaciones de startups.

---
## Visualizacion

> Antes de cada grafico: **¿qué pregunta de negocio estoy respondiendo?**

| Pregunta | Gráfico |
|---|---|
| ¿Cuántos hay por categoría? | `countplot` |
| ¿Cómo se distribuye una variable numérica? | `histplot` |
| ¿Cómo varía la distribución entre grupos? | `boxplot` |
| ¿Se relacionan dos variables numéricas? | `scatterplot` / `lmplot` |
| ¿Qué variables se mueven juntas? | `heatmap` |

### Countplot — ¿cuántos hay por categoría?

In [ ]:
# Startups por sector
orden = df_clean['sector'].value_counts().index

plt.figure(figsize=(11, 6))
sns.countplot(data=df_clean, y='sector', order=orden, palette='tab20')
plt.title('Startups de IA por sector')
plt.xlabel('Cantidad')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
df_clean['sector'].value_counts(normalize=True).round(2) * 100

In [ ]:
# Startups por pais — top 10
orden_paises = df_clean['pais'].value_counts().head(10).index

plt.figure(figsize=(10, 5))
sns.countplot(data=df_clean[df_clean['pais'].isin(orden_paises)],
              y='pais', order=orden_paises, palette='tab20')
plt.title('Top 10 paises por cantidad de startups')
plt.xlabel('Cantidad')
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
df_clean['pais'].value_counts(normalize=True).round(2) * 100

USA domina con el 71% del dataset. Israel, UK y China son los siguientes con 6%, 5% y 5% respectivamente.

`order=` fuerza el orden de mayor a menor — sin eso seaborn lo muestra en orden alfabético.

### Histograma — ¿cómo se distribuye una variable?

In [ ]:
# Distribucion del monto de ronda
media   = df_clean['monto_ronda_usd_mm'].mean()
mediana = df_clean['monto_ronda_usd_mm'].median()

plt.figure(figsize=(11, 5))
sns.histplot(data=df_clean, x='monto_ronda_usd_mm', bins=25, kde=True)
plt.axvline(media,   color='red',   linestyle='--', label=f'Media:   USD {media:,.0f}M')
plt.axvline(mediana, color='black', linestyle='--', label=f'Mediana: USD {mediana:,.0f}M')
plt.title('Distribucion del monto de ronda')
plt.xlabel('Monto (USD millones)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribucion de empleados
media_emp   = df_clean['empleados'].mean()
mediana_emp = df_clean['empleados'].median()

plt.figure(figsize=(11, 5))
sns.histplot(data=df_clean, x='empleados', bins=30, kde=True)
plt.axvline(media_emp,   color='red',   linestyle='--', label=f'Media:   {media_emp:,.0f}')
plt.axvline(mediana_emp, color='black', linestyle='--', label=f'Mediana: {mediana_emp:,.0f}')
plt.title('Distribucion de empleados')
plt.xlabel('Empleados')
plt.legend()
plt.tight_layout()
plt.show()

Ambas distribuciones tienen sesgo a la derecha — la mayoría de startups son pequeñas pero unos pocos gigantes estiran la cola.

`kde=True` dibuja la curva de densidad suavizada sobre el histograma.
`axvline` dibuja una línea vertical — útil para marcar media y mediana.

### Boxplot — ¿cómo varía la distribución entre grupos?

In [ ]:
# Valuacion por etapa de ronda
orden_rondas = ['Seed', 'Serie A', 'Serie B', 'Serie C', 'Serie D', 'Serie E', 'Serie F', 'Serie J']

plt.figure(figsize=(12, 5))
sns.boxplot(data=df_clean, x='ultima_ronda', y='valuacion_usd_mm',
            order=orden_rondas, palette='Blues')
plt.title('Valuacion por etapa de ronda')
plt.xlabel('Etapa')
plt.ylabel('Valuacion (USD millones)')
plt.tight_layout()
plt.show()

In [ ]:
# Empleados por sector — top 8
top_sectores = df_clean['sector'].value_counts().head(8).index

plt.figure(figsize=(11, 6))
sns.boxplot(data=df_clean[df_clean['sector'].isin(top_sectores)],
            x='empleados', y='sector', palette='Blues')
plt.title('Distribucion de empleados por sector')
plt.xlabel('Empleados')
plt.ylabel('')
plt.tight_layout()
plt.show()

El boxplot muestra mediana, cuartiles, rango y outliers de un vistazo. Un barplot del promedio ocultaría toda esa información.

Los puntos sueltos fuera de los bigotes son outliers — por ejemplo OpenAI en Serie E.

### Scatter — ¿se relacionan dos variables numéricas?

In [ ]:
# Monto de ronda vs valuacion
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='monto_ronda_usd_mm', y='valuacion_usd_mm',
                hue='ultima_ronda', palette='tab10', s=80, alpha=0.8)
plt.title('Monto de ronda vs Valuacion')
plt.xlabel('Monto ultima ronda (USD millones)')
plt.ylabel('Valuacion (USD millones)')
plt.legend(title='Ronda')
plt.tight_layout()
plt.show()

El punto rojo aislado arriba es OpenAI (Serie E) — USD 6.600M de ronda y USD 300.000M de valuacion. Distorsiona completamente la escala del grafico.

El punto amarillo a la derecha es Databricks (Serie J) — la ronda mas grande del dataset (USD 10.000M) pero valuacion mucho menor que OpenAI.

El 90% de las startups se concentra abajo a la izquierda — menos de USD 2.000M de ronda y menos de USD 20.000M de valuacion.

Tendencia general — a mayor monto de ronda, mayor valuación, pero la relación no es perfecta. Hay empresas que levantaron poco y valen mucho (eficiencia de capital) y viceversa.

¿que pasa si sacamos a OpenAI? ¿Se ve mejor la tendencia del resto?

In [ ]:
df_clean[['startup', 'ultima_ronda', 'monto_ronda_usd_mm', 'valuacion_usd_mm']].dropna().sort_values('valuacion_usd_mm', ascending=False).head(15)

In [ ]:
sin_openai = df_clean[df_clean['startup'] != 'OpenAI']

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sin_openai, x='monto_ronda_usd_mm', y='valuacion_usd_mm',
                hue='ultima_ronda', palette='tab10', s=80, alpha=0.8)
plt.title('Monto vs Valuacion — sin OpenAI')
plt.xlabel('Monto ultima ronda (USD millones)')
plt.ylabel('Valuacion (USD millones)')
plt.legend(title='Ronda')
plt.tight_layout()
plt.show()

Sin OpenAI la escala baja de 300.000M a 65.000M y se ve la estructura real del mercado.

Los puntos mas altos son Anthropic (Serie E, USD 60.000M) y Databricks (Serie J, USD 62.000M).
xAI aparece en Serie C con USD 50.000M — alta valuacion para una etapa relativamente temprana.

Caso interesante: Canva (Serie D) levanto solo USD 200M pero vale USD 42.000M — señal de eficiencia de capital.
Caso opuesto: Waymo (Serie B) levanto USD 5.600M y vale USD 45.000M — requirio mucho capital para llegar ahi.

Conclusion: un solo outlier (OpenAI) distorsionaba completamente la lectura. Esto pasa todo el tiempo con datos reales.

In [ ]:
# Con linea de regresion — lmplot
sns.lmplot(data=sin_openai, x='monto_ronda_usd_mm', y='valuacion_usd_mm',
           height=6, aspect=1.5,
           scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})
plt.title('Monto vs Valuacion — con regresion')
plt.xlabel('Monto ultima ronda (USD millones)')
plt.ylabel('Valuacion (USD millones)')
plt.tight_layout()
plt.show()

`scatterplot` = puntos solos. `lmplot` = puntos + línea de regresión automática.

La línea de regresión muestra la tendencia general. Si los puntos están muy dispersos, la correlación es débil.

### Heatmap — ¿qué variables se mueven juntas?

In [ ]:
# Matriz de correlacion
cols = ['anio_fundacion', 'monto_ronda_usd_mm', 'valuacion_usd_mm', 'empleados']
corr = df_clean[cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1)
plt.title('Matriz de Correlacion')
plt.tight_layout()
plt.show()

- `Verde` = correlación positiva (suben juntas)
- `Rojo` = correlación negativa (una sube, la otra baja)
- `Amarillo` = sin correlación

`monto_ronda` vs `empleados` (0.73) — la correlacion mas fuerte: las empresas que levantan mas capital tienden a tener mas empleados.

`monto_ronda` vs `valuacion` (0.66) — a mayor ronda, mayor valuacion. Tiene sentido — las rondas grandes validan valuaciones altas.

`anio_fundacion` vs `empleados` (-0.52) — las empresas mas antiguas tienen mas empleados. El signo negativo es porque anio_fundacion mas bajo = empresa mas vieja.

`anio_fundacion` vs `valuacion` (-0.19) y vs `monto_ronda` (-0.14) — correlacion debil, casi sin relacion.

In [ ]:
# Matriz de correlacion — todas las columnas numericas
corr = df_clean.select_dtypes(include='number').corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1)
plt.title('Matriz de Correlacion')
plt.tight_layout()
plt.show()